# NetraSetu - Model 1 v2: DR Severity Classifier (Branch A), retrain

**Built, not executed here.** This notebook was prepared on a dev machine with
no Kaggle access, no EyePACS data, and 16GB free disk (EyePACS alone is
80GB+) -- see the v2 training-run report for the full explanation. Run it on
Kaggle (T4/P100, EyePACS attachable as a dataset without downloading it to
your own disk), the same way `train_classifier_kaggle.ipynb` (v1) was
actually trained.

**What changed from v1, and why (v2 task items 1-5, 7):**

| # | Change | Where |
|---|---|---|
| 1 | Input resolution 384 -> **512** | `IMG_SIZE` below |
| 2 | + EyePACS (curated subset) + a second, binary referable/non-referable head | data loading, `DRClassifierV2` |
| 3 | Ordinal loss's penalty made ASYMMETRIC: under-grading costs `(true-pred)^2`, over-grading costs half that | `OrdinalWeightedCEv2` |
| 4 | Class weights: grade 4 explicitly boosted above grade 3 (v1 had this backwards: 2.34 vs 2.92, purely because grade 3 has fewer raw examples than grade 4 -- inverse-frequency weighting alone cannot know grade 4 is clinically worse to miss) | class-weight cell |
| 5 | Binary head: focal loss + EyePACS referable oversampling + a validation-ROC threshold sweep locked at >=90% sensitivity | binary-head cells |
| 7 | Evaluation reports grade-4 recall, grade-1 recall, referable sens/spec, AND QWK on the RECOVERED held-out split -- all four, every run | evaluation cell |

**Item 6** (M5 microaneurysm count -> grade-0-vs-1 decision) is NOT in this
notebook -- it's inference-time rule-engine logic, not a training change. It
was investigated with real inference on real held-out images (see
`experiments/investigateM5Grade1.py`) and NOT implemented as an automatic
grade override: a naive count threshold would have fixed at most 3/4 real
grade-1 misses while mis-escalating roughly half of a small true-grade-0
sample tested. What ships instead: `gradingOrchestrator.js`'s existing
branch-disagreement mechanism already routes this exact case (CNN=0,
rule-engine>=1) to mandatory human review, now labeled distinctly
(`grade0Vs1Disagreement`) for monitoring.

**Item 8 is NOT optional and is NOT in this notebook either** -- it happens
AFTER this notebook produces a finished, evaluated checkpoint, as a separate
MATLAB step. See this notebook's last cell and the v2 report's closing
section. `calibration_v1.json`'s `qhat=0.8432` was fitted for v1 at 384x384
and a version guard (`branchAInfer.py` / `branchAInferMatlab.m`) will now
REFUSE to apply it to a v2 model automatically -- but that guard only helps
if v2's calibration is actually re-fitted and the file actually overwritten.
It is not.

**What was fixed/hardened after the first draft (the evaluation-split fix,
EyePACS wiring, and training hygiene):**

| # | Change | Where |
|---|---|---|
| A.1-A.4 | The "recovered held-out split" is now v1's REAL saved test set (`branchA_v1_test_ids.npy`, asserted to be exactly those 628 images), not a freshly-drawn same-size split -- the old version of this notebook compared v2 against v1 on two DIFFERENT test sets without saying so. Final evaluation reports POOLED / IDRiD-only / APTOS-only, each diffed against a v1 reference number computed on that SAME population | split cell, evaluation cell |
| A.2 | `EXCLUDE_RECOVERED_VAL` (default `False`): v1's recovered VAL images return to v2's general pool by default (v1 never trained on them) instead of being held out a second time for no leakage benefit | config cell |
| B.1-B.5 | EyePACS feeds the BINARY head only (masked out of the 5-class loss and out of class-weight counting), gets a small monitor-only val slice (binary AUC, never early-stopping), a stratified curation cap (`EYEPACS_MAX`, all grade 3/4 kept) before the slow quality filter, loud failure if 0 images/rows match, and a disk-usage guard that auto-shrinks the cap before caching | data-loading + dataset/loader + class-weight cells |
| C.1 | `GRADE4_WEIGHT_BOOST` raised 1.3 -> **2.0** (1.3 only cleared grade3's weight by ~4% on v1's real counts); assertion tightened to `>= 1.25x`, not just `>` | class-weight cell |
| C.2 | `RUN_TAG` ("v2a"/"v2b", `_smoke` suffixed under `SMOKE`) in every output filename, so runs can't silently overwrite each other's artifacts | config cell |
| C.3 | `last.pt` + `history.json` written every epoch with resume-if-exists, so a killed Kaggle session doesn't restart from epoch 1 | train cell |
| C.4 | `SMOKE` flag: caps data/epochs for a fast pipeline check while keeping every leakage assertion on; only exact-count assertions are skipped | config + split cells |
| C.5 | Test report adds a "live-path" P(g3)+P(g4)>0.5 safety-check referable number next to the plain 5-class-argmax and binary-head numbers, mirroring the production rule-engine path | evaluation cell |

**Verification/robustness pass on top of that (this round):**

- The evaluation-split fix's own SMOKE ordering bug is fixed: `recovered_test_df`
  is now ALWAYS built from the full, pre-`DEBUG_MAX_PER_SOURCE` pool (moving
  `DEBUG_MAX_PER_SOURCE` to apply to the train/val `pool` only, after the
  recovered test set is carved out) -- so the exact-628 assertions hold
  unconditionally, and `SMOKE` only stratified-subsamples the ALREADY-VERIFIED
  628-row test set down to ~100, keeping a few rows of every grade present.
- `SMOKE` now reaches the last cell end-to-end: the binary-threshold ROC lock
  falls back to a loud warning + the max-sensitivity threshold instead of
  raising when 90% sensitivity is unreachable on a tiny val slice, every
  sensitivity/specificity computation is nan-safe on a zero-sample class, and
  a `SMOKE RUN COMPLETE - numbers are meaningless` banner prints at the end.
- `V1_REFERENCE_IDRID_ONLY["grade4_recall"]` was **wrong** -- it held the
  POOLED grade-4 recall (0.444) mislabeled as IDRiD-only; recomputed directly
  from `models/Model1/branchA_v1_test_logits.npy` it is **0.300** (3/10).
  `V1_REFERENCE_APTOS_ONLY` added (it didn't exist before) since the
  evaluation cell now reports an APTOS-only block too.
- EyePACS is pointed at the dataset that actually matches this pipeline --
  `tanlikesmath/diabetic-retinopathy-resized` (`resized_train/` +
  `trainLabels.csv`) -- with `EYEPACS_IMG_SUBDIR`/`EYEPACS_LABEL_CSV` pinning
  `load_eyepacs` to exactly that pair, never the `*_cropped` variant it ships
  alongside. `USE_EYEPACS` stays `False`: this only fixes the config, it does
  not turn EyePACS training on.

**v2c: weighted EyePACS 5-class contribution + domain augmentation (this round):**

| # | Change | Where |
|---|---|---|
| 1 | `EYEPACS_5CLASS_WEIGHT` (default `0.0` = today's v2b masking, byte-identical). `>0`: EyePACS rows enter the 5-class ordinal loss with per-sample weight `w_i`, `loss5 = sum(w_i*loss_i)/sum(w_i)` | config cell, `masked_ordinal_loss` |
| 2 | Class-weight counts (`counts`/`w_raw`/`w_boosted`/the `w4>=1.25*w3` assertion) now use WEIGHTED train counts (`n_k = sum of w_i` over grade-`k` train rows) -- at `EYEPACS_5CLASS_WEIGHT==0` this is numerically identical to the old non-EyePACS-only count | class-weight cell |
| 3 | EyePACS curation gets a SECOND strategy, used only when `EYEPACS_5CLASS_WEIGHT>0`: all grade 3/4, grade 2 up to 4500, ALL grade 1 (uncapped), fill with grade 0 up to `EYEPACS_MAX`. `EYEPACS_MAX` itself is unchanged as a config knob; a real v2c run sets it to 24000 | `curate_eyepacs`, `load_eyepacs` |
| 4 | `DOMAIN_AUG` (default `False` = today's pipeline, byte-identical): TRAIN-ONLY appearance augmentation (per-channel gain, blur, noise, down-up resolution loss) on the cached Ben-Graham image, before the existing geometric transforms. `SMOKE` mode saves an 8-sample montage to `/kaggle/working/aug_check.png` | dataset cell |
| 5 | Monitor-only: EyePACS-val 5-class QWK, logged when `EYEPACS_5CLASS_WEIGHT>0`, never used for early stopping/selection | metrics cell, train cell |
| 6 | `RUN_TAG`: `"v2a"` (no EyePACS) / `"v2b"` (EyePACS, `EYEPACS_5CLASS_WEIGHT==0`) / `"v2c"` (EyePACS, `EYEPACS_5CLASS_WEIGHT>0`), `_smoke` suffixed under `SMOKE` as before | config cell |

Every new/changed piece defaults to reproducing today's v2b behaviour
EXACTLY -- `EYEPACS_5CLASS_WEIGHT=0.0` and `DOMAIN_AUG=False` are both the
defaults, and the v2b code PATHS (not just their numeric output) are left
untouched, with the v2c behaviour added as an early branch ahead of them.
Ben Graham preprocessing, `DRClassifierV2`, `OrdinalWeightedCEv2`'s formula,
`FocalLossBCE`, `IMG_SIZE=512`, the optimizer/scheduler, the checkpoint
schema (fields only added, none removed/renamed), the binary-threshold lock
fix, `SMOKE` behaviour, and every leakage assertion are all unchanged by
this round. **Patched, not executed here** -- same as every prior round of
this notebook.

## Mandatory pre-flight: exclude the recovered held-out set

`models/Model1/branchA_v1_{test,val}_{ids,labels}.npy` are v1's OWN saved
split arrays -- the only trustworthy record of which images were genuinely
held out, because they were written at split time, before any model existed
to leak into them. Upload them as a small private Kaggle dataset alongside
APTOS/IDRiD/EyePACS and set `RECOVERED_SPLIT_DIR` below. **v1's TEST ids are
always dropped from v2's training AND validation data** (see
`EXCLUDE_RECOVERED_VAL` for the separate VAL-id decision), even though v2
draws from a larger, freshly-combined pool -- otherwise a v1 held-out image
could land in v2's training set by chance, and any v1-vs-v2 comparison on it
would be silently invalid. `experiments/recoverHeldOutSplit.py` implements
and sanity-checks this same exclusion set on the dev machine; the cell below
reimplements only the load + exclude step (not the sanity checks, which need
local file access this notebook won't have on Kaggle).


In [ ]:
import os

# =====================================================================
#  CONFIG
# =====================================================================
IDRID_ROOT = "/kaggle/input/idrid-disease-grading"
APTOS_ROOT = "/kaggle/input/aptos2019-blindness-detection"

# EyePACS: tanlikesmath/diabetic-retinopathy-resized (a Kaggle re-upload of
# the original "Diabetic Retinopathy Detection" competition data, resized to
# <=1024px and already extracted -- avoids the raw competition's multi-part
# zip problem). Ships resized_train/ + resized_train_cropped/ + trainLabels.csv
# + trainLabels_cropped.csv side by side; EYEPACS_IMG_SUBDIR/EYEPACS_LABEL_CSV
# below pin load_eyepacs to exactly the uncropped pair and nothing else.
EYEPACS_ROOT = "/kaggle/input/diabetic-retinopathy-resized"
EYEPACS_IMG_SUBDIR = "resized_train"
EYEPACS_LABEL_CSV = "trainLabels.csv"
USE_EYEPACS = False  # this pass only fixes EyePACS's config/loader to point
                      # at the right dataset -- it does NOT turn EyePACS
                      # training on. Flip to True deliberately for a real
                      # v2b run. False -> v2 architecture/loss changes only,
                      # no new data (still useful: isolates items 1/3/4 from
                      # items 2/5)
EYEPACS_MAX = 10000  # item B.4: curated cap, not a raw random sample. With
                      # EYEPACS_5CLASS_WEIGHT==0 (default): today's v2b
                      # curation -- ALL grade 3/4 kept, grade 2 capped ~2500,
                      # remainder filled with grades 0/1 (seeded). With
                      # EYEPACS_5CLASS_WEIGHT>0: v2c curation -- ALL grade
                      # 3/4, grade 2 capped 4500, ALL grade 1, remainder
                      # filled with grade 0 ONLY (see curate_eyepacs). A real
                      # v2c run sets EYEPACS_MAX=24000 here (before the
                      # quality filter, which kept only 62% of EyePACS in the
                      # v2b run -- about 15k should survive). May be reduced
                      # automatically by the disk guard (item B.5) if free
                      # disk is tight -- that guard reads EYEPACS_MAX directly
                      # so it stays correct at any cap size, v2c's included.

# v2c item 1: EyePACS's contribution to the 5-CLASS ordinal loss.
#   0.0 (default) = today's v2b behaviour EXACTLY: EyePACS is masked out of
#        the 5-class loss entirely (masked_ordinal_loss's hard exclusion),
#        and out of the class-weight counts (item 2) -- the unchanged v2b
#        CODE PATH runs, not just a numerically-equal replacement.
#   >0.0 = v2c: EyePACS rows enter the 5-class loss with per-sample weight
#        w_i=EYEPACS_5CLASS_WEIGHT (APTOS/IDRiD always w_i=1.0):
#        loss5 = sum(w_i * loss_i) / sum(w_i) over the batch. Also changes
#        the EyePACS curation strategy (see EYEPACS_MAX above) and the
#        class-weight counts (item 2: n_k = sum of w_i over train rows of
#        grade k, not a plain count).
EYEPACS_5CLASS_WEIGHT = 0.0

# v2c item 4: TRAIN-ONLY domain/appearance augmentation (per-channel gain,
# blur, noise, down-up resolution loss), applied to the cached Ben-Graham
# image BEFORE the existing geometric transforms -- see the dataset cell's
# apply_domain_augmentation(). False (default) = today's pipeline, bit-
# identical (the function is never called). A real v2c run sets this True.
DOMAIN_AUG = False

# v1's own recovered held-out split -- upload branchA_v1_{test,val}_{ids,labels}.npy
# as a private Kaggle dataset. v1's TEST ids are ALWAYS excluded from v2
# training; see EXCLUDE_RECOVERED_VAL below for the VAL ids.
RECOVERED_SPLIT_DIR = "/kaggle/input/branch-a-v1-recovered-split"
EXCLUDE_RECOVERED_VAL = False  # item A.2: False -> v1's recovered VAL images
                                # return to v2's general train/val pool (v1
                                # never trained on them, and holding them out
                                # too costs v2 ~30% fewer grade-4 cases than
                                # v1 itself trained on, for no leakage benefit
                                # -- only the TEST ids matter for that). True
                                # -> exclude VAL too (old behavior, matches
                                # the pre-fix version of this notebook).

USE_IDRID = True
DEBUG_MAX_PER_SOURCE = None   # e.g. 200 for a fast end-to-end pipeline check
                               # (set automatically by SMOKE below, or set
                               # this directly). Applied to the TRAIN/VAL pool
                               # only, AFTER the recovered test set is carved
                               # out -- see the split cell -- so it can never
                               # shrink or invalidate the recovered test set.
SMOKE = False  # <<< FLIP TO True FOR A SMOKE RUN, False FOR THE REAL RUN >>>
                # item C.4: fast end-to-end pipeline check. Caps data volume,
                # epochs, and the recovered-test size (see below and the
                # split cell); EVERY leakage assertion (train/val/test/
                # eyepacs-val disjoint, no recovered-TEST id in train or val)
                # stays ON -- only the exact-628 / exact-count assertions are
                # skipped. Also (verification pass): lets the binary
                # threshold lock fall back to a warning instead of raising,
                # and suffixes RUN_TAG with "_smoke" so these outputs can
                # never be mistaken for a real run's.

OUT_DIR = "/kaggle/working"
CACHE_DIR = "/kaggle/temp/bg_cache_512"

# ---- item 1: resolution ----
# 512, not 640. Chosen, not defaulted: every other model in this pipeline
# (M2 vessel, M3 localization, M4/M5 lesions) already runs at 512, so this
# keeps one "resolution family" across the whole system rather than adding a
# second, and it is still a large jump from 384 (~1.78x linear, ~3.16x pixel
# count) -- expected to be the largest single lever on the grade-3/4 boundary
# specifically, per the v2 task's own diagnosis (384px too coarse to resolve
# fine neovascular fronds). 640 costs materially more VRAM/time for a
# resolution nothing else here uses; if 512 does not move grade-4 recall
# enough on the recovered held-out set (see the evaluation cell), sweep 640
# next rather than assuming it would have helped.
IMG_SIZE = 512

SEED = 42
BATCH_SIZE = 24          # smaller than v1's 32 -- 512px costs ~1.78x the
                          # activation memory of 384px per image; raise this
                          # if your GPU has headroom (T4/P100: 16GB, should
                          # comfortably fit 24-32 at 512 for EfficientNet-B0)
EPOCHS = 30
PATIENCE = 7
LR = 1e-4
WEIGHT_DECAY = 1e-5
DROP_RATE = 0.3
NUM_WORKERS = 4
USE_AMP = True
MODEL_NAME = "efficientnet_b0"
NUM_CLASSES = 5

if SMOKE:
    DEBUG_MAX_PER_SOURCE = 200
    EPOCHS = 2
    print(f"SMOKE=True -> DEBUG_MAX_PER_SOURCE={DEBUG_MAX_PER_SOURCE}, EPOCHS={EPOCHS}, "
          f"recovered test stratified-subsampled to ~100 (see split cell). Leakage "
          f"assertions stay ON; exact-count assertions are skipped.")

# item 4 / C.1: explicit grade-4-over-grade-3 boost. See the class-weight
# cell for why plain inverse-frequency weighting gets this backwards. 1.3x
# only lifted grade 4 ~4% above grade 3 on v1's actual counts -- not enough
# margin to trust across runs with slightly different counts, hence 2.0x and
# a tightened assertion (grade4 weight >= 1.25x grade3 weight, not just >).
GRADE4_WEIGHT_BOOST = 2.0

# item 5: binary head
REFERABLE_FROM = 2
FOCAL_GAMMA = 2.0
BINARY_LOSS_WEIGHT = 0.5   # relative to the 5-class ordinal loss; both are
                            # logged separately every epoch so this can be
                            # retuned without re-deriving it from scratch
TARGET_SENSITIVITY = 0.90  # item 5's locked operating point

# item C.2: tag every output filename with which config produced it, so a
# v2a (no EyePACS) and v2b (with EyePACS) run never silently overwrite each
# other's artifacts in the same Kaggle "Output" panel -- and a SMOKE run
# never gets mistaken for a real one.
# v2c item 6: "v2a" (no EyePACS) / "v2b" (EyePACS, 5-class-masked -- today's
# behaviour) / "v2c" (EyePACS contributing to the 5-class loss).
if not USE_EYEPACS:
    RUN_TAG = "v2a"
elif EYEPACS_5CLASS_WEIGHT == 0:
    RUN_TAG = "v2b"
else:
    RUN_TAG = "v2c"
if SMOKE:
    RUN_TAG += "_smoke"

CKPT_PATH         = f"{OUT_DIR}/branchA_{RUN_TAG}.pt"
LAST_CKPT_PATH    = f"{OUT_DIR}/branchA_{RUN_TAG}_last.pt"
HISTORY_PATH      = f"{OUT_DIR}/branchA_{RUN_TAG}_history.json"
VAL_LOGITS_PATH   = f"{OUT_DIR}/branchA_{RUN_TAG}_val_logits.npy"
VAL_LABELS_PATH   = f"{OUT_DIR}/branchA_{RUN_TAG}_val_labels.npy"
VAL_IDS_PATH      = f"{OUT_DIR}/branchA_{RUN_TAG}_val_ids.npy"
VAL_BIN_LOGITS_PATH = f"{OUT_DIR}/branchA_{RUN_TAG}_val_binary_logits.npy"
TEST_LOGITS_PATH  = f"{OUT_DIR}/branchA_{RUN_TAG}_test_logits.npy"
TEST_LABELS_PATH  = f"{OUT_DIR}/branchA_{RUN_TAG}_test_labels.npy"
TEST_IDS_PATH     = f"{OUT_DIR}/branchA_{RUN_TAG}_test_ids.npy"
TEST_BIN_LOGITS_PATH = f"{OUT_DIR}/branchA_{RUN_TAG}_test_binary_logits.npy"
METRICS_PATH      = f"{OUT_DIR}/branchA_{RUN_TAG}_metrics.json"

print("IDRID_ROOT   :", IDRID_ROOT, "| exists:", os.path.isdir(IDRID_ROOT))
print("APTOS_ROOT   :", APTOS_ROOT, "| exists:", os.path.isdir(APTOS_ROOT))
print("EYEPACS_ROOT :", EYEPACS_ROOT, "| exists:", os.path.isdir(EYEPACS_ROOT),
      "| USE_EYEPACS:", USE_EYEPACS, "| EYEPACS_MAX:", EYEPACS_MAX,
      "| EYEPACS_5CLASS_WEIGHT:", EYEPACS_5CLASS_WEIGHT, "| DOMAIN_AUG:", DOMAIN_AUG)
print("EYEPACS_IMG_SUBDIR:", EYEPACS_IMG_SUBDIR, "| EYEPACS_LABEL_CSV:", EYEPACS_LABEL_CSV)
print("RECOVERED_SPLIT_DIR:", RECOVERED_SPLIT_DIR, "| exists:", os.path.isdir(RECOVERED_SPLIT_DIR),
      "| EXCLUDE_RECOVERED_VAL:", EXCLUDE_RECOVERED_VAL)
print("RUN_TAG:", RUN_TAG, "| SMOKE:", SMOKE)


In [ ]:
import os, json, random, time, glob, warnings, shutil
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.transforms as T
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (cohen_kappa_score, f1_score, confusion_matrix,
                             roc_curve, recall_score, roc_auc_score)

warnings.filterwarnings("ignore", category=UserWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| timm", timm.__version__, "| device:", device)
if device.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0), f"| {p.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected - enable Settings -> Accelerator -> GPU T4")

try:
    from torch.amp import autocast as _ac, GradScaler as _GS
    def amp_autocast():
        return _ac("cuda", enabled=USE_AMP and device.type == "cuda")
    def make_scaler():
        return _GS("cuda", enabled=USE_AMP and device.type == "cuda")
except Exception:
    from torch.cuda.amp import autocast as _ac, GradScaler as _GS
    def amp_autocast():
        return _ac(enabled=USE_AMP and device.type == "cuda")
    def make_scaler():
        return _GS(enabled=USE_AMP and device.type == "cuda")

def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    s = torch.initial_seed() % (2 ** 32)
    np.random.seed(s)
    random.seed(s)

seed_everything(SEED)
g = torch.Generator()
g.manual_seed(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print("cache dir :", CACHE_DIR)
print("output dir:", OUT_DIR)


## 1. Ben Graham preprocessing (inlined, UNCHANGED from v1)

Identical to `preprocessing/ben_graham.py` and to v1's own copy of this cell
-- only `target_size` (now 512 by default via `IMG_SIZE`) differs, and that
was already a parameter, not a hardcoded value. Do not "improve" this
function; see `preprocessModel1.m`'s header for why a changed preprocessing
recipe is a silent accuracy regression, not an improvement, once a model has
trained against a specific one.

In [ ]:
def ben_graham_preprocess(image: np.ndarray, target_size: int = 384) -> np.ndarray:
    """Ben Graham-style fundus preprocessing. Identical to preprocessing/ben_graham.py.

    image       : BGR uint8 array, shape (H, W, 3)
    target_size : output square side length
    returns     : uint8 array (target_size, target_size, 3), values 0..255, BGR order
    """
    if image is None or image.size == 0:
        raise ValueError("ben_graham_preprocess received an empty or None image.")
    if image.ndim != 3 or image.shape[2] != 3:
        raise ValueError(f"Expected a 3-channel image, got shape {image.shape}.")

    gray = image[:, :, 1]
    _, mask = cv2.threshold(gray, 7, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        x, y = max(0, x), max(0, y)
        w = min(w, image.shape[1] - x)
        h = min(h, image.shape[0] - y)
        cropped = image[y:y + h, x:x + w]
    else:
        cropped = image
    if cropped.size == 0:
        cropped = image

    resized = cv2.resize(cropped, (target_size, target_size), interpolation=cv2.INTER_AREA)

    sigma = target_size / 30.0
    ksize = max(int(sigma) * 2 + 1, 1)
    blurred = cv2.GaussianBlur(resized, (ksize, ksize), sigma)
    enhanced = cv2.addWeighted(resized, 4, blurred, -4, 128)
    return enhanced


_t = np.zeros((512, 512, 3), np.uint8)
cv2.circle(_t, (256, 256), 200, (40, 180, 60), -1)
_o = ben_graham_preprocess(_t, IMG_SIZE)
print("self-test OK - output", _o.shape, _o.dtype, "range", int(_o.min()), int(_o.max()))


## 2. Data loading: APTOS + IDRiD (unchanged) + EyePACS (new, item 2)

EyePACS's own Kaggle "Diabetic Retinopathy Detection" competition labels
(`0`=No DR .. `4`=Proliferative DR) are ALREADY on the same ICDR 0-4 scale
APTOS and IDRiD use -- "ICDR-harmonized" here is mostly verification, not
remapping: `load_eyepacs` asserts the label column only contains `{0..4}`
and fails loudly if it does not, rather than silently coercing something
that turns out not to match.

"Curated/quality-filtered subset": EyePACS is large and famously noisy
(off-center, out-of-focus, or artifact-heavy photos are common in the raw
competition set). `eyepacs_quality_ok` is a lightweight OpenCV filter, not a
port of the project's own MATLAB quality gate (that only runs in MATLAB, and
this notebook has no MATLAB) -- it rejects images that are too dark/bright on
average, too low-contrast (a cheap blur proxy via Laplacian variance), or
implausibly small. Thresholds are marked as a starting point, not fitted
numbers -- re-tune them against a labeled sample of EyePACS images actually
rejected/kept before trusting this at scale, the same way `redFloor`/
`grade3QuadMin` in `ruleEngineGrade.m` were fitted against real held-out
images rather than picked by feel.

In [ ]:
def eyepacs_quality_ok(bgr, min_mean=15, max_mean=240, min_lap_var=15.0, min_side=256):
    """Cheap, disclosed-as-unfitted quality gate for raw EyePACS images.

    Not the project's real quality gate (MATLAB-only, not portable here).
    Rejects: too dark, too bright/washed-out, too blurry (low Laplacian
    variance -- the same focus proxy phc-local-app's assessFocus.m uses,
    just not the same fitted threshold), or too small to be a real fundus
    photo rather than a thumbnail/corrupt file.
    """
    if bgr is None or bgr.size == 0:
        return False
    h, w = bgr.shape[:2]
    if min(h, w) < min_side:
        return False
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    mean_val = float(gray.mean())
    if mean_val < min_mean or mean_val > max_mean:
        return False
    lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    if lap_var < min_lap_var:
        return False
    return True


def load_aptos(root):
    csv = os.path.join(root, "train.csv")
    img_dir = os.path.join(root, "train_images")
    if not os.path.isfile(csv):
        raise FileNotFoundError(f"APTOS train.csv not found at {csv}.")
    df = pd.read_csv(csv).rename(columns={"id_code": "image_id", "diagnosis": "grade"})
    df["path"] = df["image_id"].apply(lambda s: os.path.join(img_dir, f"{s}.png"))
    df["source"] = "aptos"
    df["grade"] = pd.to_numeric(df["grade"], errors="coerce")
    df = df.dropna(subset=["grade"])
    df["grade"] = df["grade"].astype(int)
    return df[["image_id", "path", "grade", "source"]]


def _tag_of(path):
    low = path.lower()
    return "test" if "test" in low else ("train" if "train" in low else "all")


def load_idrid(root):
    if not os.path.isdir(root):
        raise FileNotFoundError(f"IDRID_ROOT '{root}' does not exist.")
    imgs = []
    for e in ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG", "*.png"):
        imgs += glob.glob(os.path.join(root, "**", e), recursive=True)
    by_tag_stem, by_stem = {}, {}
    for p in imgs:
        stem = os.path.splitext(os.path.basename(p))[0]
        by_tag_stem.setdefault((_tag_of(p), stem), p)
        by_stem.setdefault(stem, p)

    rows = []
    for c in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
        try:
            t = pd.read_csv(c)
        except Exception:
            continue
        t.columns = [str(x).strip() for x in t.columns]
        gcol = next((col for col in t.columns
                     if col.lower().replace("_", " ").startswith("retinopathy grade")), None)
        ncol = next((col for col in t.columns
                     if col.lower().replace("_", " ").startswith("image name")), None)
        if not (gcol and ncol):
            continue
        ctag = _tag_of(c)
        t = t[[ncol, gcol]].rename(columns={ncol: "name", gcol: "grade"})
        t["name"] = t["name"].astype(str).str.strip()
        t["grade"] = pd.to_numeric(t["grade"], errors="coerce")
        t = t.dropna(subset=["name", "grade"])
        for name, grade in zip(t["name"], t["grade"]):
            p = by_tag_stem.get((ctag, name)) or by_stem.get(name)
            if p is None:
                continue
            rows.append({"image_id": f"idrid_{ctag}_{name}", "path": p,
                         "grade": int(grade), "source": "idrid"})
    if not rows:
        raise FileNotFoundError(f"No IDRiD grade CSV rows matched under {root}.")
    df = pd.DataFrame(rows).drop_duplicates(subset="image_id").reset_index(drop=True)
    return df[["image_id", "path", "grade", "source"]]


def curate_eyepacs(df, max_images, seed, use_v2c_strategy=False):
    """item B.4 (v2b, use_v2c_strategy=False, DEFAULT -- today's behaviour,
    UNCHANGED code path below): keep ALL grade 3/4 (rarest, clinically most
    important), sample grade 2 up to ~2500, fill the remainder with grades
    0/1 combined (seeded).

    v2c item 3 (use_v2c_strategy=True, when EYEPACS_5CLASS_WEIGHT>0): keep
    ALL grade 3/4, grade 2 up to 4500, ALL grade 1 (uncapped -- EyePACS now
    feeds the 5-class loss, so its grade-1 volume is worth keeping in full
    rather than competing with grade 0 for a shared budget), fill the
    remainder with grade 0 ONLY up to max_images.

    Runs BEFORE the slow, per-image quality filter so no decode/quality-check
    work is wasted on images that would be dropped by curation anyway.
    """
    if max_images is None or len(df) <= max_images:
        return df
    if max_images <= 0:
        return df.iloc[0:0]
    g34 = df[df["grade"].isin([3, 4])]
    if len(g34) >= max_images:
        print(f"WARNING: EyePACS grade 3+4 alone ({len(g34)}) already meets/exceeds "
              f"EYEPACS_MAX ({max_images}) -- keeping only a {max_images}-sized sample "
              f"of grade 3/4, dropping grades 0/1/2 entirely.")
        return g34.sample(max_images, random_state=seed).reset_index(drop=True)

    if use_v2c_strategy:
        g2 = df[df["grade"] == 2]
        g2_keep = g2.sample(min(len(g2), 4500), random_state=seed)
        g1_keep = df[df["grade"] == 1]   # ALL of grade 1, uncapped
        required = pd.concat([g34, g2_keep, g1_keep])
        if len(required) >= max_images:
            print(f"WARNING: EyePACS grade 3+4 + grade2(<=4500) + ALL grade 1 "
                  f"({len(required)}) already meets/exceeds EYEPACS_MAX ({max_images}) "
                  f"-- keeping only a {max_images}-sized sample of that pool; grade 0 "
                  f"gets none.")
            return required.sample(max_images, random_state=seed).reset_index(drop=True)
        budget = max_images - len(required)
        g0 = df[df["grade"] == 0]
        g0_keep = g0.sample(min(len(g0), budget), random_state=seed) if budget > 0 else g0.iloc[0:0]
        return (pd.concat([required, g0_keep])
                .sample(frac=1, random_state=seed).reset_index(drop=True))

    budget = max_images - len(g34)
    g2 = df[df["grade"] == 2]
    g2_keep = g2.sample(min(len(g2), 2500), random_state=seed)
    budget -= len(g2_keep)
    g01 = df[df["grade"].isin([0, 1])]
    g01_keep = g01.sample(min(len(g01), max(budget, 0)), random_state=seed) if budget > 0 else g01.iloc[0:0]
    return (pd.concat([g34, g2_keep, g01_keep])
            .sample(frac=1, random_state=seed).reset_index(drop=True))


def load_eyepacs(root, img_subdir=EYEPACS_IMG_SUBDIR, label_csv_name=EYEPACS_LABEL_CSV,
                 quality_filter=True, max_images=None, use_v2c_curation=False):
    """EyePACS via tanlikesmath/diabetic-retinopathy-resized: exactly
    `label_csv_name` (columns image, level) matched against images under
    exactly `root/img_subdir` -- NEVER the *_cropped/ variant, and never a
    different CSV that happens to also have "label" in its name (this
    dataset ships trainLabels.csv AND trainLabels_cropped.csv side by side).
    quality_filter=True applies eyepacs_quality_ok per-image -- this reads
    every file once (slow) but there is no metadata field to filter on
    instead; a real quality label was never collected for this dataset.
    """
    if not os.path.isdir(root):
        raise FileNotFoundError(
            f"EYEPACS_ROOT '{root}' does not exist. Attach the "
            f"tanlikesmath/diabetic-retinopathy-resized dataset via Kaggle's Add Data panel.")
    label_csv = os.path.join(root, label_csv_name)
    if not os.path.isfile(label_csv):
        raise FileNotFoundError(
            f"Expected the EyePACS label CSV at exactly {label_csv} -- this notebook only "
            f"ever reads '{label_csv_name}', never a *_cropped.csv or any other CSV under {root}.")
    img_dir = os.path.join(root, img_subdir)
    if not os.path.isdir(img_dir):
        raise FileNotFoundError(
            f"Expected the EyePACS image folder at exactly {img_dir} -- this notebook only "
            f"ever reads images from '{img_subdir}/', never the *_cropped/ variant.")

    t = pd.read_csv(label_csv)
    t.columns = [c.strip().lower() for c in t.columns]
    # tanlikesmath/diabetic-retinopathy-resized uses "image","level" exactly;
    # the fallback list only covers other naming schemes should this
    # dataset's own column names ever change.
    img_col = next((c for c in t.columns if c in ("image", "id_code", "image_id")), t.columns[0])
    lvl_col = next((c for c in t.columns if c in ("level", "diagnosis", "grade")), t.columns[1])
    t = t.rename(columns={img_col: "image_id", lvl_col: "grade"})
    t["grade"] = pd.to_numeric(t["grade"], errors="coerce")
    t = t.dropna(subset=["grade"])
    t["grade"] = t["grade"].astype(int)
    # item 2: ICDR harmonization is a VERIFICATION, not a remap -- EyePACS
    # already uses 0-4. Fail loudly rather than silently coercing an
    # unexpected label set into range.
    bad = set(t["grade"].unique()) - {0, 1, 2, 3, 4}
    assert not bad, f"EyePACS grade column has out-of-ICDR-range values: {bad} -- verify the label mapping before proceeding."

    img_files = {}
    for e in ("*.jpg", "*.jpeg", "*.png"):
        for p in glob.glob(os.path.join(img_dir, "**", e), recursive=True):
            img_files[os.path.splitext(os.path.basename(p))[0]] = p
    # item B.3: fail loudly rather than silently training on an empty/near-empty set.
    assert len(img_files) > 0, (
        f"Found {label_csv_name} but 0 image files (*.jpg/*.jpeg/*.png) under {img_dir}. "
        f"Attach the tanlikesmath/diabetic-retinopathy-resized dataset PRE-EXTRACTED "
        f"(it ships resized_train/ + resized_train_cropped/ + both label CSVs already "
        f"extracted, not as zips) and make sure '{img_subdir}/' is actually populated.")
    t["path"] = t["image_id"].astype(str).map(img_files)
    t = t.dropna(subset=["path"]).reset_index(drop=True)
    assert len(t) > 0, (
        f"Found {len(img_files)} image file(s) under {img_dir} but 0 rows matched an "
        f"image_id in {label_csv_name} -- check that '{img_subdir}/' and '{label_csv_name}' "
        f"actually correspond to each other (don't mix resized_train/ with "
        f"trainLabels_cropped.csv, or vice versa).")

    if max_images is not None:
        n_before_curation = len(t)
        t = curate_eyepacs(t, max_images, SEED, use_v2c_strategy=use_v2c_curation)
        curation_desc = ("all grade 3/4, grade 2 up to 4500, ALL grade 1, fill remainder with grade 0"
                         if use_v2c_curation else
                         "all grade 3/4, up to 2500 grade 2, fill remainder with grades 0/1")
        print(f"EyePACS curation: {n_before_curation} -> {len(t)} (target {max_images}: {curation_desc}, seeded)")

    if quality_filter:
        keep = []
        for p in tqdm(t["path"], desc="EyePACS quality filter", mininterval=5.0):
            img = cv2.imread(p)
            keep.append(eyepacs_quality_ok(img))
        n_before = len(t)
        t = t[np.array(keep)].reset_index(drop=True)
        print(f"EyePACS quality filter: kept {len(t)}/{n_before} "
              f"({100*len(t)/max(n_before,1):.1f}%) -- thresholds are a documented "
              f"starting point (see markdown above), not fitted.")

    # item 3: final per-grade composition, AFTER the quality filter -- the
    # composition curation targeted may not survive the filter unchanged
    # (grade-dependent rejection rates are plausible), so this is what
    # actually enters training, not just what curation asked for.
    print("EyePACS final per-grade composition (after quality filter):",
          t["grade"].value_counts().reindex(range(5), fill_value=0).sort_index().to_dict())

    t["source"] = "eyepacs"
    return t[["image_id", "path", "grade", "source"]]


frames = [load_aptos(APTOS_ROOT)]
if USE_IDRID:
    frames.append(load_idrid(IDRID_ROOT))

if USE_EYEPACS:
    # item B.5: disk guard BEFORE loading EyePACS -- estimate the on-disk
    # Ben Graham cache size for everything that will need caching (APTOS +
    # IDRiD already loaded above, plus up to EYEPACS_MAX EyePACS images) and
    # shrink EYEPACS_MAX automatically if that would eat more than 90% of
    # free disk, rather than failing mid-cache with a full disk.
    n_non_eyepacs = sum(len(f) for f in frames)
    est_total_images = n_non_eyepacs + EYEPACS_MAX
    est_bytes = est_total_images * (IMG_SIZE ** 2) * 3
    free_bytes = shutil.disk_usage(CACHE_DIR).free
    print(f"disk guard: estimated cache size {est_bytes / 1e9:.2f} GB for up to "
          f"~{est_total_images} images at {IMG_SIZE}x{IMG_SIZE} | "
          f"free disk: {free_bytes / 1e9:.2f} GB")
    if est_bytes > 0.9 * free_bytes:
        max_affordable_total = int((0.9 * free_bytes) / ((IMG_SIZE ** 2) * 3))
        new_eyepacs_max = max(0, max_affordable_total - n_non_eyepacs)
        print(f"WARNING: estimated cache size would exceed 90% of free disk -- "
              f"reducing EYEPACS_MAX {EYEPACS_MAX} -> {new_eyepacs_max}")
        EYEPACS_MAX = new_eyepacs_max
    frames.append(load_eyepacs(EYEPACS_ROOT, EYEPACS_IMG_SUBDIR, EYEPACS_LABEL_CSV, max_images=EYEPACS_MAX,
                               use_v2c_curation=(EYEPACS_5CLASS_WEIGHT > 0)))

data = pd.concat(frames, ignore_index=True)
data = data[data["grade"].between(0, 4)].reset_index(drop=True)
data["image_id"] = data["source"] + "__" + data["image_id"].astype(str)

exists = data["path"].apply(os.path.isfile)
if (~exists).any():
    print(f"WARNING: {(~exists).sum()} rows point to a missing image file - dropped")
data = data[exists].reset_index(drop=True)

# NOTE: DEBUG_MAX_PER_SOURCE is deliberately NOT applied here. It is applied
# in the split cell, to the post-exclusion train/val `pool` only -- never to
# `data` itself -- so that `recovered_test_df` (built from `data` in the next
# cell) is always the FULL, real recovered test set, never subsampled. See
# the split cell for why (SMOKE mode used to build recovered_test_df from an
# already-shrunk pool, which broke the exact-628 assertion).
print("\nTotal usable images (pre held-out exclusion, pre DEBUG_MAX_PER_SOURCE):", len(data))
print(pd.crosstab(data["source"], data["grade"], margins=True))


## 3. Recovered held-out split -- v1's REAL test set, always excluded from
training; the same cell also carves out EyePACS's own small monitor-only
val slice (item B.1.2)

`image_id` here is already `<source>__<id>` (built above), matching
`branchA_v1_{test,val}_ids.npy`'s own id format exactly -- see
`experiments/recoverHeldOutSplit.py` for how those were produced and
verified.

**This replaces an earlier version of this cell that produced an invalid
v1-vs-v2 comparison:** it excluded v1's recovered VAL+TEST ids from the pool,
built a BRAND NEW 70/15/15 split, and then reported that new split's test set
as if it were "the recovered held-out split" -- it wasn't; it was a
different, freshly-drawn ~628-image set that only happened to be the same
size. Now:

- `recovered_test_df` is built directly from v1's saved `branchA_v1_test_ids.npy`
  against the PRE-exclusion, PRE-`DEBUG_MAX_PER_SOURCE` pool (`data`), and is
  asserted to be exactly those 628 images -- proven present by an exact count
  check across ALL sources, not just IDRiD, so the same assertion also proves
  APTOS's content-hash ids matched. It is the ONLY test set anywhere in this
  notebook, and these assertions hold UNCONDITIONALLY (SMOKE included) since
  `data` is never subsampled before this point.
- `EXCLUDE_RECOVERED_VAL` (config cell) controls whether v1's recovered VAL
  images return to v2's general train/val pool (`False`, default -- v1 never
  trained on them, and excluding them too would cost v2 ~30% of the grade-4
  cases v1 itself trained on) or are excluded as well, matching the old
  behavior (`True`).
- `DEBUG_MAX_PER_SOURCE` (including via `SMOKE`) is applied HERE, to `pool`
  (the train/val pool, after recovered ids are excluded) -- never to `data`,
  and never to `recovered_test_df`. Under `SMOKE`, `recovered_test_df` is
  instead separately, stratified-subsampled down to ~100 rows afterwards,
  guaranteeing a few rows of every grade present in the real 628.
- The remaining APTOS+IDRiD pool is split 85/15 train/val, stratified by
  grade, per source. `val_df` is APTOS+IDRiD only -- it drives early
  stopping, temperature scaling, conformal calibration, and the binary
  threshold lock.
- EyePACS gets no test split and no role in that APTOS/IDRiD val split at
  all: 95% goes straight into `train_df` (binary head only, item B.1), 5%
  becomes `eyepacs_val_df`, a monitor-only slice used only to log binary AUC.


In [ ]:
recovered_test_ids = set(np.load(os.path.join(RECOVERED_SPLIT_DIR, "branchA_v1_test_ids.npy"), allow_pickle=True))
recovered_val_ids = set(np.load(os.path.join(RECOVERED_SPLIT_DIR, "branchA_v1_val_ids.npy"), allow_pickle=True))
print(f"recovered v1 TEST ids: {len(recovered_test_ids)} | recovered v1 VAL ids: {len(recovered_val_ids)}")

# item A.1 (fixed ordering): recovered_test_df is built from `data`, which is
# ALWAYS the full pool -- DEBUG_MAX_PER_SOURCE (including via SMOKE) is only
# ever applied below to `pool`, AFTER this. That means these exact-count
# assertions hold UNCONDITIONALLY, SMOKE or not; the recovered test set is
# never accidentally shrunk before it's verified.
recovered_test_present = recovered_test_ids & set(data["image_id"])
print(f"recovered TEST ids found in this combined pool: {len(recovered_test_present)}/{len(recovered_test_ids)}")
assert len(recovered_test_ids) == 628, (
    f"expected v1's recovered TEST id file to contain exactly 628 ids, found "
    f"{len(recovered_test_ids)} -- wrong file attached as RECOVERED_SPLIT_DIR?")
assert len(recovered_test_present) == len(recovered_test_ids), (
    f"only {len(recovered_test_present)}/{len(recovered_test_ids)} of v1's recovered "
    f"TEST ids were found in the combined APTOS+IDRiD(+EyePACS) pool. This is a hard "
    f"stop: if ids don't match exactly (including APTOS's content-hash ids), "
    f"'recovered_test_df' below is NOT actually v1's real held-out test set, and "
    f"every v1-vs-v2 comparison downstream would be invalid. Fix the id-construction "
    f"scheme (compare against experiments/recoverHeldOutSplit.py) before proceeding.")

recovered_test_df = data[data["image_id"].isin(recovered_test_ids)].reset_index(drop=True)
assert len(recovered_test_df) == 628, (
    f"recovered_test_df has {len(recovered_test_df)} rows, expected exactly 628")


def _stratified_smoke_subsample(df, n_total, seed, min_per_grade=3):
    """SMOKE-only: keep at least `min_per_grade` rows (or all available) for
    EVERY grade present in `df`, then fill the remaining budget up to
    n_total at random -- so a tiny SMOKE test set still exercises every
    grade the real recovered test set has (grade1_recall/grade4_recall would
    otherwise come back nan just from unlucky pure-random sampling, not from
    a real bug)."""
    guaranteed_parts = [d.sample(min(len(d), min_per_grade), random_state=seed)
                        for _, d in df.groupby("grade")]
    guaranteed = pd.concat(guaranteed_parts) if guaranteed_parts else df.iloc[0:0]
    remaining = df.drop(guaranteed.index)
    extra_n = max(n_total - len(guaranteed), 0)
    extra = remaining.sample(min(len(remaining), extra_n), random_state=seed) if extra_n > 0 else remaining.iloc[0:0]
    return pd.concat([guaranteed, extra]).sample(frac=1, random_state=seed).reset_index(drop=True)


if SMOKE:
    recovered_test_df = _stratified_smoke_subsample(recovered_test_df, 100, SEED)
    print(f"SMOKE=True -> recovered test (already verified as the real 628) stratified-"
          f"subsampled to {len(recovered_test_df)} (every grade present in the full 628 "
          f"keeps >= a few rows, not pure-random)")

# item A.2: v1's TEST ids are ALWAYS excluded from train/val -- not optional.
# Its VAL ids are excluded too only if EXCLUDE_RECOVERED_VAL=True (old
# behavior); by default (False) they return to the general pool, since v1
# never trained on them and holding them out costs v2 real grade-4 volume
# for no leakage benefit.
excluded_ids = set(recovered_test_ids) | (recovered_val_ids if EXCLUDE_RECOVERED_VAL else set())
val_note = f"+ {len(recovered_val_ids)} VAL" if EXCLUDE_RECOVERED_VAL else ", VAL returned to pool"
print(f"EXCLUDE_RECOVERED_VAL={EXCLUDE_RECOVERED_VAL} -> excluding {len(excluded_ids)} ids from "
      f"train/val ({len(recovered_test_ids)} TEST always{val_note})")

pool = data[~data["image_id"].isin(excluded_ids)].reset_index(drop=True)
print(f"remaining pool for v2 train/val split: {len(pool)}")

if DEBUG_MAX_PER_SOURCE:
    # item A.1 fix: DEBUG_MAX_PER_SOURCE (including via SMOKE) shrinks the
    # TRAIN/VAL pool only -- recovered_test_df above was already built and
    # verified from the FULL pool and is completely unaffected by this.
    pool = pd.concat([
        d.sample(min(len(d), DEBUG_MAX_PER_SOURCE), random_state=SEED)
        for _, d in pool.groupby("source")
    ]).reset_index(drop=True)
    print(f"DEBUG_MAX_PER_SOURCE={DEBUG_MAX_PER_SOURCE} -> train/val pool downsampled to "
          f"{len(pool)} images (recovered_test_df is UNAFFECTED)")


def split_85_15(df, seed=SEED):
    idx = np.arange(len(df))
    y = df["grade"].values
    tr, va = train_test_split(idx, test_size=0.15, random_state=seed, stratify=y)
    return df.iloc[tr], df.iloc[va]


# item A.3 + B.1: APTOS/IDRiD get an 85/15 stratified-by-grade, per-source
# split (val_df drives early stopping, temperature scaling, conformal
# calibration, and the binary threshold lock). EyePACS gets no test split
# and no say in that val_df -- 95% goes straight to train_df (binary head
# only), 5% becomes a small monitor-only eyepacs_val_df (item B.1.2).
parts_train, parts_val = [], []
eyepacs_val_df = pool.iloc[0:0]
for src, d in pool.groupby("source"):
    d = d.reset_index(drop=True)
    if src == "eyepacs":
        tr_idx, va_idx = train_test_split(np.arange(len(d)), test_size=0.05,
                                          random_state=SEED, stratify=d["grade"].values)
        parts_train.append(d.iloc[tr_idx])
        eyepacs_val_df = d.iloc[va_idx].reset_index(drop=True)
        print(f"{src:8s} -> train {len(tr_idx):5d} | monitor-only val {len(va_idx):4d} | (no test split)")
    else:
        tr, va = split_85_15(d, SEED)
        parts_train.append(tr)
        parts_val.append(va)
        print(f"{src:8s} -> train {len(tr):5d} | val {len(va):4d}")

train_df = pd.concat(parts_train).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(parts_val).reset_index(drop=True) if parts_val else pool.iloc[0:0]  # APTOS+IDRiD only
test_df = recovered_test_df  # v1's REAL recovered test -- the only test set (item A.3)

a, b, c = set(train_df.image_id), set(val_df.image_id), set(test_df.image_id)
e = set(eyepacs_val_df.image_id)
assert not (a & b) and not (a & c) and not (b & c), "split leakage detected (train/val/test)"
assert not (a & e) and not (b & e) and not (c & e), "split leakage detected (eyepacs monitor-val)"
assert not (a & recovered_test_ids), "a recovered TEST id leaked into v2 TRAINING -- stop and fix before spending compute"
assert not (b & recovered_test_ids), "a recovered TEST id leaked into v2 VAL -- stop and fix before spending compute"

print(f"\nCOMBINED  train {len(train_df)} | val {len(val_df)} (APTOS+IDRiD only) | "
      f"test {len(test_df)} (v1 recovered) | eyepacs monitor-val {len(eyepacs_val_df)}")
print("\ntrain grade x source:")
print(pd.crosstab(train_df.source, train_df.grade, margins=True))


## 4. Ben Graham preprocess + cache to disk (unchanged pattern, new resolution)

In [ ]:
def cache_key(source, image_id):
    return f"{source}_{image_id}_{IMG_SIZE}"


def cache_path_for(source, image_id):
    return os.path.join(CACHE_DIR, cache_key(source, image_id) + ".npy")


def _preprocess_and_cache(row):
    out_path = cache_path_for(row.source, row.image_id)
    if os.path.isfile(out_path):
        return True
    bgr = cv2.imread(row.path, cv2.IMREAD_COLOR)
    if bgr is None:
        return False
    proc = ben_graham_preprocess(bgr, IMG_SIZE)
    rgb = cv2.cvtColor(proc, cv2.COLOR_BGR2RGB)
    np.save(out_path, rgb)
    return True


all_rows = pd.concat([train_df, val_df, test_df, eyepacs_val_df], ignore_index=True)
with ThreadPoolExecutor(max_workers=8) as ex:
    results = list(tqdm(ex.map(_preprocess_and_cache, [r for _, r in all_rows.iterrows()]),
                        total=len(all_rows), desc="ben_graham cache", mininterval=5.0))
n_failed = sum(1 for ok in results if not ok)
print(f"cached {len(results) - n_failed}/{len(results)} images at {IMG_SIZE}x{IMG_SIZE} ({n_failed} failed to read)")


## 5. Dataset + augmentation (unchanged from v1 except IMG_SIZE, and the
binary referable label added alongside the 5-class one)

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1111)),
    T.RandomRotation(20, fill=0),
    T.RandomHorizontalFlip(0.5),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def apply_domain_augmentation(arr):
    """v2c item 4: TRAIN-ONLY appearance/domain augmentation, applied to the
    cached Ben-Graham uint8 RGB image BEFORE the existing geometric
    transforms (train_tf) -- simulates portable/off-brand camera variation
    (gain, focus, sensor noise, resolution) the public training sources
    (APTOS/IDRiD/EyePACS, all professional fundus cameras) do not exhibit.
    Each of the four perturbations is applied independently at its own
    probability -- not mutually exclusive, a single image can get any
    combination including all four or none. Uses the GLOBAL numpy random
    state (seeded by seed_everything/seed_worker, the same discipline the
    rest of this pipeline's augmentation already relies on), not a fresh
    unseeded generator, so DataLoader worker reproducibility is unaffected.

    Only ever called when DOMAIN_AUG=True (RetinaDataset below); with
    DOMAIN_AUG=False (default) this function is never invoked and the
    pipeline is bit-identical to before.
    """
    x = arr.astype(np.float32)

    # (a) per-channel gain g_c ~ U(0.7,1.3), independent per channel, p=0.8
    if np.random.random() < 0.8:
        gains = np.random.uniform(0.7, 1.3, size=3).astype(np.float32)
        x = (x - 128.0) * gains[np.newaxis, np.newaxis, :] + 128.0
        x = np.clip(x, 0, 255)

    # (b) Gaussian blur, sigma ~ U(0.3,1.6), p=0.4
    if np.random.random() < 0.4:
        sigma = float(np.random.uniform(0.3, 1.6))
        ksize = max(3, int(2 * round(3 * sigma) + 1))  # odd, ~3 sigma support
        x = cv2.GaussianBlur(x, (ksize, ksize), sigma)

    # (c) additive Gaussian noise, sigma ~ U(2,8) intensity levels, p=0.3
    if np.random.random() < 0.3:
        sigma = float(np.random.uniform(2.0, 8.0))
        noise = np.random.normal(0.0, sigma, size=x.shape).astype(np.float32)
        x = np.clip(x + noise, 0, 255)

    # (d) down-up scaling: resize to s~U(0.5,0.9) INTER_AREA, back up to the
    # original size INTER_LINEAR -- simulates a lower-resolution capture, p=0.3
    if np.random.random() < 0.3:
        h, w = x.shape[:2]
        s = float(np.random.uniform(0.5, 0.9))
        nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
        down = cv2.resize(x.astype(np.uint8), (nw, nh), interpolation=cv2.INTER_AREA)
        x = cv2.resize(down, (w, h), interpolation=cv2.INTER_LINEAR).astype(np.float32)

    return np.clip(x, 0, 255).astype(np.uint8)


def save_domain_aug_montage(df, n=8, path="/kaggle/working/aug_check.png", seed=SEED):
    """SMOKE-mode-only visual sanity check for DOMAIN_AUG: tiles `n` samples,
    each with apply_domain_augmentation freshly applied to its cached
    Ben-Graham image, into one montage PNG. Purely diagnostic -- does not
    affect training, and is never called when DOMAIN_AUG=False."""
    rows = df.sample(min(n, len(df)), random_state=seed)
    panels = []
    for _, r in rows.iterrows():
        arr = np.load(cache_path_for(r.source, r.image_id))
        aug = apply_domain_augmentation(arr)
        bgr = cv2.cvtColor(aug, cv2.COLOR_RGB2BGR)
        panel = cv2.resize(bgr, (256, 256))
        cv2.putText(panel, f"{r.source}:{int(r.grade)}", (6, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        panels.append(panel)
    cols = 4
    rows_n = (len(panels) + cols - 1) // cols
    canvas = np.zeros((rows_n * 256, cols * 256, 3), dtype=np.uint8)
    for i, p in enumerate(panels):
        rr, cc = divmod(i, cols)
        canvas[rr * 256:(rr + 1) * 256, cc * 256:(cc + 1) * 256] = p
    cv2.imwrite(path, canvas)
    print(f"domain-aug montage ({len(panels)} samples) saved to {path}")


class RetinaDataset(Dataset):
    def __init__(self, df, transform, domain_aug=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.domain_aug = domain_aug  # v2c item 4: TRAIN ONLY -- callers
                                       # must pass domain_aug=False explicitly
                                       # for val/test/eyepacs-val datasets

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        arr = np.load(cache_path_for(r.source, r.image_id))
        if self.domain_aug:
            arr = apply_domain_augmentation(arr)
        img = self.transform(Image.fromarray(arr))
        grade = int(r.grade)
        referable = float(grade >= REFERABLE_FROM)
        is_eyepacs = float(r.source == "eyepacs")  # item B.1: which loss
                                                     # terms this sample may
                                                     # contribute to
        return img, grade, referable, is_eyepacs, cache_key(r.source, r.image_id)


train_ds = RetinaDataset(train_df, train_tf, domain_aug=DOMAIN_AUG)  # v2c item 4: TRAIN ONLY
val_ds = RetinaDataset(val_df, eval_tf, domain_aug=False)
test_ds = RetinaDataset(test_df, eval_tf, domain_aug=False)

# item 5: oversample referable EyePACS cases specifically. EyePACS's own
# class imbalance is more severe than APTOS/IDRiD's (heavily grade-0
# dominant), and it is the largest source in the combined pool once
# attached, so weighting the SAMPLER (not just the loss) keeps referable
# cases from being drowned out epoch-to-epoch, not just down-weighted in a
# loss term that still sees them rarely.
sample_weights = np.ones(len(train_df), dtype=np.float64)
is_eyepacs_referable = (train_df["source"].values == "eyepacs") & (train_df["grade"].values >= REFERABLE_FROM)
if is_eyepacs_referable.any():
    referable_count = int(is_eyepacs_referable.sum())
    nonreferable_count = len(train_df) - referable_count
    boost = nonreferable_count / max(referable_count, 1)
    sample_weights[is_eyepacs_referable] = boost
    print(f"EyePACS referable oversampling: {referable_count} referable cases "
          f"boosted {boost:.2f}x in the training sampler")
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_df), replacement=True)

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True,
                 worker_init_fn=seed_worker, persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(train_ds, sampler=sampler, drop_last=True, generator=g, **loader_kw)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kw)
test_loader = DataLoader(test_ds, shuffle=False, **loader_kw)

# item B.1.2: EyePACS's own small (5%) monitor-only val slice -- binary AUC
# only, logged every epoch, never used for early stopping, checkpoint
# selection, or the locked binary threshold (all of those stay APTOS+IDRiD
# only via val_loader above).
if len(eyepacs_val_df) > 0:
    eyepacs_val_ds = RetinaDataset(eyepacs_val_df, eval_tf, domain_aug=False)
    eyepacs_val_loader = DataLoader(eyepacs_val_ds, shuffle=False, **loader_kw)
    print(f"eyepacs monitor-only val: {len(eyepacs_val_df)} images (binary AUC only, not used for early stopping)")
else:
    eyepacs_val_loader = None

xb, yb, rb, eb, idb = next(iter(train_loader))
print("sample batch:", tuple(xb.shape), xb.dtype, "| grades", yb[:8].tolist(),
      "| referable", rb[:8].tolist(), "| is_eyepacs", eb[:8].tolist())

# v2c item 4: visual sanity check, SMOKE mode only (and only meaningful when
# DOMAIN_AUG is actually on -- nothing new to show otherwise).
if SMOKE and DOMAIN_AUG:
    save_domain_aug_montage(train_df, n=8)


## 6. Model - EfficientNet-B0, TWO heads (item 2)

Same backbone family as v1 (`timm` EfficientNet-B0, ImageNet-pretrained),
retrained at 512px. `headBin` is new: a second linear layer reading the SAME
pooled+dropout features as the 5-class head, trained jointly. Sharing the
trunk (rather than a fully separate binary model) is deliberate -- both
heads are answering questions about the same underlying pathology signal,
and a shared trunk means the binary head benefits from EyePACS's larger
referable-case volume even though the 5-class head is still evaluated
primarily on APTOS+IDRiD's more reliable ICDR grading.

In [ ]:
class DRClassifierV2(nn.Module):
    """timm backbone (feature mode) + explicit Dropout + TWO linear heads.

    self.head5   - 5-class ICDR grade (unchanged role from v1's DRClassifier)
    self.headBin - binary referable/non-referable (item 2), one logit,
                   BCEWithLogitsLoss-compatible.

    The explicit nn.Dropout before both heads is still what MC-Dropout
    (Task 6.1) toggles at inference -- unchanged from v1, and load-bearing
    for the SAME reason: timm's own head dropout is functional, not a
    module, and MC-Dropout needs a module to force into train() mode.
    """
    def __init__(self, model_name, num_classes, drop_rate, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained,
                                          num_classes=0, drop_rate=0.0)
        self.num_features = self.backbone.num_features
        self.drop = nn.Dropout(p=drop_rate)
        self.head5 = nn.Linear(self.num_features, num_classes)
        self.headBin = nn.Linear(self.num_features, 1)

    def forward(self, x):
        feats = self.drop(self.backbone(x))
        return self.head5(feats), self.headBin(feats).squeeze(-1)


ARCH_DESC = ("DRClassifierV2: timm(model_name, num_classes=0, drop_rate=0) -> "
             "nn.Dropout(drop_rate) -> {head5: Linear(num_features,5), "
             "headBin: Linear(num_features,1)}")

try:
    model = DRClassifierV2(MODEL_NAME, NUM_CLASSES, DROP_RATE, pretrained=True).to(device)
except Exception as e:
    raise RuntimeError(
        "Could not create the pretrained model. Turn Settings -> Internet -> On so timm "
        "can download ImageNet weights, then re-run this cell.") from e

n_params = sum(p.numel() for p in model.parameters()) / 1e6
n_drop = sum(1 for m in model.modules() if isinstance(m, nn.Dropout) and m.p > 0)
print(f"{MODEL_NAME} (dual head): {n_params:.1f}M params | dropout modules: {n_drop} | img_size {IMG_SIZE}")
assert n_drop >= 1, "expected a real nn.Dropout before the heads for MC-Dropout"


## 7. Losses (items 3, 4, 5)

Three pieces, reported separately every epoch so a regression in one doesn't
hide inside a single combined number:

1. **Class weights (item 4).** Inverse-frequency weighting alone gets grade
   4 vs grade 3 backwards whenever grade 4 happens to have MORE raw training
   examples than grade 3 (true for v1: 250 vs 200) -- the formula only
   compensates for imbalance, it has no idea grade 4 is clinically worse to
   miss. `GRADE4_WEIGHT_BOOST` is applied ON TOP of inverse-frequency
   weighting, specifically and only to grade 4, and the cell asserts the
   result actually reversed the ordering before training starts.

2. **`OrdinalWeightedCEv2` (item 3).** v1's ordinal factor was
   `1 + |pred - target|` -- symmetric, so over- and under-grading by the same
   amount cost the same. v2's is asymmetric: `1 + (true-pred)^2` when the
   model UNDER-grades (pred < true), `1 + 0.5*(true-pred)^2` when it
   OVER-grades or is exact. Squared rather than linear specifically to
   penalize LARGE under-grading errors (grade 0 predicted for a true grade 4)
   much more than small ones (grade 3 predicted for a true grade 4) -- this
   is the loss-side lever aimed at recovering grade-4 recall, aligned with
   QWK's own quadratic penalty instead of leaving CE and the selection metric
   mismatched (the v2 task's own framing).

3. **`FocalLossBCE` (item 5).** Standard focal loss on the binary head,
   `gamma=FOCAL_GAMMA`, combined with the class-balanced sampler already
   built into `train_loader` above (oversampling handles the volume
   imbalance; focal loss handles the easy-negative-dominance imbalance within
   each batch -- the two address different things and neither substitutes for
   the other).

In [ ]:
# item B.2 (v2b, EYEPACS_5CLASS_WEIGHT==0, DEFAULT): class weights use
# non-EyePACS train counts ONLY -- EyePACS never contributes to the 5-class
# loss, so it must not skew the counts that weight it either. This is the
# EXACT current behaviour, reproduced below as the EYEPACS_5CLASS_WEIGHT==0
# special case of the general weighted formula (a plain count IS a sum of
# w_i=1.0 over included rows and w_i=0.0 over excluded ones).
#
# v2c item 2 (EYEPACS_5CLASS_WEIGHT>0): n_k = sum of w_i over train rows of
# grade k, w_i=EYEPACS_5CLASS_WEIGHT for EyePACS rows, 1.0 for APTOS/IDRiD --
# so EyePACS now contributes ITS weighted share to the class-balance
# computation, consistent with its new weighted contribution to the loss
# itself (masked_ordinal_loss below).
sample_weight_5class = np.where(train_df["source"].values == "eyepacs",
                                float(EYEPACS_5CLASS_WEIGHT), 1.0)
counts = (pd.Series(sample_weight_5class, index=train_df.index)
          .groupby(train_df["grade"]).sum()
          .reindex(range(NUM_CLASSES), fill_value=0.0).sort_index())
total = float(counts.sum())
w_raw = total / (NUM_CLASSES * counts.clip(lower=1).astype(float))

w_boosted = w_raw.copy()
w_boosted[4] = w_raw[4] * GRADE4_WEIGHT_BOOST

counts_label = ("non-EyePACS" if EYEPACS_5CLASS_WEIGHT == 0
               else f"weighted, EyePACS w={EYEPACS_5CLASS_WEIGHT}")
print(f"train grade counts ({counts_label}):", {k: round(v, 2) for k, v in counts.to_dict().items()})
print("inverse-freq weights            :", {k: round(v, 3) for k, v in w_raw.to_dict().items()})
print("grade4-boosted weights          :", {k: round(v, 3) for k, v in w_boosted.to_dict().items()})
print(f"grade3 weight {w_raw[3]:.3f} vs grade4 weight {w_boosted[4]:.3f} "
      f"(boost x{GRADE4_WEIGHT_BOOST}) -> grade4 >= 1.25*grade3: {w_boosted[4] >= 1.25 * w_raw[3]}")
assert w_boosted[4] >= 1.25 * w_raw[3], (
    "GRADE4_WEIGHT_BOOST is not large enough to put grade 4 meaningfully above grade 3 "
    "against THIS run's actual class counts (item C.1: a boost that only just clears "
    "grade4>grade3, e.g. 1.3x on v1's counts, was ~4% -- not enough margin to trust). "
    "Raise GRADE4_WEIGHT_BOOST and re-check before training rather than weakening this "
    "assertion. Do not train with grade 4 weighted below grade 3; that was the bug being fixed.")

class_weights = torch.tensor(w_boosted.values, dtype=torch.float32, device=device)


class OrdinalWeightedCEv2(nn.Module):
    """Inverse-freq + grade4-boosted weighted CE, scaled per-sample by an
    ASYMMETRIC ordinal factor -- see markdown above for the exact formula
    and why it is squared and asymmetric. UNCHANGED formula; `reduction`
    added (v2c item 1) so a caller can get PER-SAMPLE losses to apply its
    own weighting (masked_ordinal_loss below) instead of this module's own
    mean -- reduction="mean" (the default, and the only mode any pre-v2c
    call site used) is byte-identical to before."""
    def __init__(self, class_weights):
        super().__init__()
        self.register_buffer("cw", class_weights.detach().clone())

    def forward(self, logits, target, reduction="mean"):
        ce = F.cross_entropy(logits, target, weight=self.cw, reduction="none")
        pred = logits.detach().argmax(dim=1)
        diff = (target - pred).float()          # true - pred, signed
        under = diff > 0                         # predicted LOWER than true
        penalty = torch.where(under, diff.pow(2), 0.5 * diff.pow(2))
        factor = 1.0 + penalty
        per_sample = ce * factor
        if reduction == "none":
            return per_sample
        return per_sample.mean()


class FocalLossBCE(nn.Module):
    """Binary focal loss on raw logits (numerically stable via
    binary_cross_entropy_with_logits, not sigmoid then BCE)."""
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(
            logits, target, reduction="none", pos_weight=self.pos_weight)
        p = torch.sigmoid(logits)
        p_t = p * target + (1 - p) * (1 - target)
        focal = bce * (1 - p_t).pow(self.gamma)
        return focal.mean()


def masked_ordinal_loss(criterion5, logits5, labels, is_eyepacs, eyepacs_weight=0.0):
    """EyePACS's contribution to the 5-class ordinal loss, controlled by
    `eyepacs_weight` (pass EYEPACS_5CLASS_WEIGHT at every call site).

    eyepacs_weight<=0 (DEFAULT, item B.1 / v2b, UNCHANGED): EyePACS rows are
    excluded entirely via a hard mask -- the exact code path this function
    always ran before v2c existed. Guards the all-EyePACS-batch case (the
    eyepacs_val monitor loader) by returning a zero loss instead of calling
    criterion5 on an empty tensor.

    eyepacs_weight>0 (v2c item 1): every row contributes, weighted --
    w_i=eyepacs_weight for EyePACS rows, 1.0 for APTOS/IDRiD --
    loss5 = sum(w_i * loss_i) / sum(w_i) over the batch. Guards a
    zero-total-weight batch (e.g. genuinely empty) the same way.
    """
    if eyepacs_weight <= 0:
        keep = ~is_eyepacs.bool()
        if not keep.any():
            return torch.zeros((), device=logits5.device, dtype=logits5.dtype)
        return criterion5(logits5[keep], labels[keep])

    weights = torch.where(
        is_eyepacs.bool(),
        torch.full_like(is_eyepacs, float(eyepacs_weight), dtype=logits5.dtype),
        torch.ones_like(is_eyepacs, dtype=logits5.dtype),
    )
    denom = weights.sum()
    if denom <= 0:
        return torch.zeros((), device=logits5.device, dtype=logits5.dtype)
    per_sample = criterion5(logits5, labels, reduction="none")
    return (weights * per_sample).sum() / denom


criterion5 = OrdinalWeightedCEv2(class_weights).to(device)
criterionBin = FocalLossBCE(gamma=FOCAL_GAMMA).to(device)
eyepacs_5class_desc = ("non-EyePACS only" if EYEPACS_5CLASS_WEIGHT == 0
                      else f"all sources, EyePACS w={EYEPACS_5CLASS_WEIGHT}")
print(f"\nlosses ready: OrdinalWeightedCEv2 (5-class, {eyepacs_5class_desc}) + "
      f"FocalLossBCE gamma={FOCAL_GAMMA} (binary, all sources)")
print(f"combined as: {1.0} * ordinal_loss + {BINARY_LOSS_WEIGHT} * focal_loss")


## 8. Metrics (item 7: ALL FOUR, every epoch and at final evaluation)

QWK alone is what the v2 task explicitly warns against checking in
isolation: it is insensitive to 54 grade-4 images inside a 628-image test
set and would hide a grade-4-recall regression completely. `compute_metrics`
below always returns all four: QWK, referable sensitivity/specificity,
grade-4 recall, grade-1 recall -- and the training loop and final evaluation
both print all four, not a subset.

In [ ]:
def _safe_rate(numerator, denom):
    """nan (not a silently-wrong 0.0) when the denominator is 0 -- e.g. a
    SMOKE-subsampled split with zero true-referable or zero true-non-
    referable cases. Used everywhere a sens/spec-style rate is computed."""
    return float("nan") if denom == 0 else numerator / denom


def compute_metrics(y_true, y_pred, referable_from=REFERABLE_FROM):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=list(range(NUM_CLASSES)))

    ref_true = y_true >= referable_from
    ref_pred = y_pred >= referable_from
    tp = int((ref_true & ref_pred).sum())
    fn = int((ref_true & ~ref_pred).sum())
    tn = int((~ref_true & ~ref_pred).sum())
    fp = int((~ref_true & ref_pred).sum())
    ref_sens = _safe_rate(tp, tp + fn)
    ref_spec = _safe_rate(tn, tn + fp)

    g4_mask = y_true == 4
    g4_recall = recall_score(y_true[g4_mask], y_pred[g4_mask], labels=[4], average="micro") if g4_mask.any() else float("nan")
    g1_mask = y_true == 1
    g1_recall = recall_score(y_true[g1_mask], y_pred[g1_mask], labels=[1], average="micro") if g1_mask.any() else float("nan")

    mean_signed_error = float((y_pred - y_true).mean())  # negative = systematic under-grading

    return {
        "qwk": float(qwk), "macro_f1": float(macro_f1),
        "ref_sens": float(ref_sens), "ref_spec": float(ref_spec),
        "grade4_recall": float(g4_recall), "grade4_n": int(g4_mask.sum()),
        "grade1_recall": float(g1_recall), "grade1_n": int(g1_mask.sum()),
        "mean_signed_error": mean_signed_error,
    }


@torch.no_grad()
def evaluate(model, loader, criterion5, criterionBin):
    model.eval()
    all_logits5, all_logits_bin, all_labels, all_ref, all_ids = [], [], [], [], []
    run_loss5, run_lossBin, seen = 0.0, 0.0, 0
    for imgs, labels, referable, is_eyepacs, ids in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels_t = labels.to(device, non_blocking=True)
        ref_t = referable.to(device, non_blocking=True).float()
        is_eyepacs_t = is_eyepacs.to(device, non_blocking=True)
        with amp_autocast():
            logits5, logitsBin = model(imgs)
            l5 = masked_ordinal_loss(criterion5, logits5, labels_t, is_eyepacs_t,
                                     eyepacs_weight=EYEPACS_5CLASS_WEIGHT)
            lb = criterionBin(logitsBin, ref_t)
        run_loss5 += l5.item() * imgs.size(0)
        run_lossBin += lb.item() * imgs.size(0)
        seen += imgs.size(0)
        all_logits5.append(logits5.float().cpu().numpy())
        all_logits_bin.append(logitsBin.float().cpu().numpy())
        all_labels.append(labels.numpy())
        all_ref.append(referable.numpy())
        all_ids.extend(ids)

    logits5 = np.concatenate(all_logits5)
    logits_bin = np.concatenate(all_logits_bin)
    labels = np.concatenate(all_labels)
    y_pred = logits5.argmax(axis=1)
    m = compute_metrics(labels, y_pred)
    m["loss5"] = run_loss5 / max(seen, 1)
    m["lossBin"] = run_lossBin / max(seen, 1)
    m["loss"] = m["loss5"] + BINARY_LOSS_WEIGHT * m["lossBin"]
    return m, logits5, logits_bin, labels, np.array(all_ids)


@torch.no_grad()
def eyepacs_monitor_auc(model, loader):
    """item B.1: EyePACS's monitor-only val split gets no 5-class metrics --
    just the binary head's AUC, logged every epoch but never used for early
    stopping, checkpoint selection, or the locked binary threshold."""
    if loader is None or len(loader.dataset) == 0:
        return float("nan")
    model.eval()
    all_probs, all_true = [], []
    for imgs, labels, referable, is_eyepacs, ids in loader:
        imgs = imgs.to(device, non_blocking=True)
        with amp_autocast():
            _, logitsBin = model(imgs)
        all_probs.append(torch.sigmoid(logitsBin).float().cpu().numpy())
        all_true.append(referable.numpy())
    probs = np.concatenate(all_probs)
    true = np.concatenate(all_true)
    if len(np.unique(true)) < 2:
        return float("nan")
    return float(roc_auc_score(true, probs))


@torch.no_grad()
def eyepacs_monitor_5class_qwk(model, loader):
    """v2c item 5 (monitor-only): 5-class QWK on EyePACS's own monitor-only
    val slice, logged every epoch when EYEPACS_5CLASS_WEIGHT>0 (EyePACS now
    contributes to the 5-class loss, so its own held-out slice is worth
    watching) -- NEVER used for early stopping, checkpoint selection, or the
    locked binary threshold, exactly like eyepacs_monitor_auc above."""
    if loader is None or len(loader.dataset) == 0:
        return float("nan")
    model.eval()
    all_logits5, all_labels = [], []
    for imgs, labels, referable, is_eyepacs, ids in loader:
        imgs = imgs.to(device, non_blocking=True)
        with amp_autocast():
            logits5, _ = model(imgs)
        all_logits5.append(logits5.float().cpu().numpy())
        all_labels.append(labels.numpy())
    logits5 = np.concatenate(all_logits5)
    labels = np.concatenate(all_labels)
    if len(labels) < 2:
        return float("nan")
    y_pred = logits5.argmax(axis=1)
    return float(cohen_kappa_score(labels, y_pred, weights="quadratic"))


## 9. Train

In [ ]:
seed_everything(SEED)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = make_scaler()

start_epoch = 1
best_qwk, best_epoch, no_improve = -1.0, -1, 0
history = []

# item C.3: resume-if-exists. Kaggle sessions get preempted/time out;
# LAST_CKPT_PATH is written every epoch (unlike CKPT_PATH, which only
# updates on a new best QWK) specifically so a killed run can pick back up
# mid-training instead of restarting from epoch 1 and burning compute twice.
if os.path.isfile(LAST_CKPT_PATH):
    last = torch.load(LAST_CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(last["model_state_dict"])
    optimizer.load_state_dict(last["optimizer_state_dict"])
    scheduler.load_state_dict(last["scheduler_state_dict"])
    scaler.load_state_dict(last["scaler_state_dict"])
    start_epoch = last["epoch"] + 1
    best_qwk = last["best_qwk"]
    best_epoch = last.get("best_epoch", -1)
    history = last.get("history", [])
    no_improve = (start_epoch - 1 - best_epoch) if best_epoch > 0 else (start_epoch - 1)
    print(f"resuming from {LAST_CKPT_PATH}: last completed epoch {last['epoch']} "
          f"-> starting at epoch {start_epoch} | best QWK so far {best_qwk:.4f} "
          f"(epoch {best_epoch}) | no_improve={no_improve}")

print(f"train {len(train_df)} | val {len(val_df)} | batch {BATCH_SIZE} | img {IMG_SIZE} | "
      f"up to {EPOCHS} epochs | early stop patience {PATIENCE} | starting at epoch {start_epoch}\n")

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    run_loss, seen, t0 = 0.0, 0, time.time()
    pbar = tqdm(train_loader, desc=f"epoch {epoch:02d}/{EPOCHS} [train]", leave=False, mininterval=5.0)
    for imgs, labels, referable, is_eyepacs, _ in pbar:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        referable = referable.to(device, non_blocking=True).float()
        is_eyepacs = is_eyepacs.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with amp_autocast():
            logits5, logitsBin = model(imgs)
            loss5 = masked_ordinal_loss(criterion5, logits5, labels, is_eyepacs,
                                        eyepacs_weight=EYEPACS_5CLASS_WEIGHT)
            lossBin = criterionBin(logitsBin, referable)
            loss = loss5 + BINARY_LOSS_WEIGHT * lossBin
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item() * imgs.size(0)
        seen += imgs.size(0)
        pbar.set_postfix(loss=f"{run_loss / seen:.4f}")
    scheduler.step()
    train_loss = run_loss / max(seen, 1)

    vm, v_logits5, v_logits_bin, v_labels, v_ids = evaluate(model, val_loader, criterion5, criterionBin)
    eyepacs_auc = eyepacs_monitor_auc(model, eyepacs_val_loader)
    # v2c item 5 (monitor-only): EyePACS-val 5-class QWK, only meaningful
    # once EyePACS actually feeds the 5-class loss; nan otherwise (v2a/v2b),
    # and never used for early stopping / checkpoint selection either way.
    eyepacs_5class_qwk = (eyepacs_monitor_5class_qwk(model, eyepacs_val_loader)
                          if EYEPACS_5CLASS_WEIGHT > 0 else float("nan"))
    lr_now = optimizer.param_groups[0]["lr"]
    dt = time.time() - t0
    improved = vm["qwk"] > best_qwk + 1e-5
    flag = "   <-- new best" if improved else ""

    # ALL FOUR metrics printed every epoch (item 7) -- not just QWK, so a
    # grade-4/grade-1 regression is visible during training, not discovered
    # only at the end. eyepacs_val_auc is monitor-only (item B.1): never used
    # for early stopping or checkpoint selection, just logged so a collapse
    # there is visible without waiting for a full report.
    eyepacs_qwk_str = f" | eyepacs_val_5class_qwk {eyepacs_5class_qwk:.3f}" if EYEPACS_5CLASS_WEIGHT > 0 else ""
    print(f"epoch {epoch:02d}/{EPOCHS} | {dt:5.0f}s | lr {lr_now:.2e} | "
          f"train_loss {train_loss:.4f} | val_loss {vm['loss']:.4f} | "
          f"QWK {vm['qwk']:.4f} | refDR sens {vm['ref_sens']:.3f} spec {vm['ref_spec']:.3f} | "
          f"grade4_recall {vm['grade4_recall']:.3f} (n={vm['grade4_n']}) | "
          f"grade1_recall {vm['grade1_recall']:.3f} (n={vm['grade1_n']}) | "
          f"eyepacs_val_auc {eyepacs_auc:.3f}{eyepacs_qwk_str}{flag}")

    if epoch == start_epoch:
        # verification-pass item 4: a quick "will this finish in time" signal
        # right after the first epoch actually run this session (not
        # necessarily epoch 1, if resuming).
        projected_total_s = dt * EPOCHS
        print(f"  epoch time: {dt:.0f}s/epoch -> projected total for {EPOCHS} epochs: "
              f"{projected_total_s / 60:.1f} min ({projected_total_s:.0f}s)")

    history.append({"epoch": epoch, "train_loss": float(train_loss), "lr": float(lr_now),
                    "seconds": float(dt), "eyepacs_val_auc": float(eyepacs_auc),
                    "eyepacs_val_5class_qwk": float(eyepacs_5class_qwk), **vm})
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2, default=float)

    if improved:
        best_qwk, best_epoch, no_improve = vm["qwk"], epoch, 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "model_name": MODEL_NAME,
            "arch": ARCH_DESC,
            "num_classes": NUM_CLASSES,
            "num_features": int(model.num_features),
            "drop_rate": DROP_RATE,
            "img_size": IMG_SIZE,   # <-- 512, not 384. This is the field
                                     # branchAInfer.py / branchAInferMatlab.m's
                                     # calibration version-guard checks.
            "normalize_mean": list(IMAGENET_MEAN),
            "normalize_std": list(IMAGENET_STD),
            "channel_order": "RGB",
            "preprocessing": "ben_graham: circular crop -> resize -> gaussian-subtraction contrast",
            "class_weights": [float(x) for x in class_weights.detach().cpu().tolist()],
            "grade4_weight_boost": GRADE4_WEIGHT_BOOST,
            # v2c item 2: WEIGHTED counts (n_k = sum of w_i over train rows of
            # grade k) -- at EYEPACS_5CLASS_WEIGHT==0 every value is a whole
            # number identical to the old non-EyePACS-only count, just typed
            # as float instead of truncated with int() (which would silently
            # discard the fractional weighting a v2c run actually needs).
            "train_grade_counts": {int(k): round(float(v), 4) for k, v in counts.to_dict().items()},
            "referable_from": REFERABLE_FROM,
            "binary_loss_weight": BINARY_LOSS_WEIGHT,
            "focal_gamma": FOCAL_GAMMA,
            "epoch": epoch,
            "val_qwk": float(best_qwk),
            "val_metrics": vm,
            "seed": SEED,
            # v2c: NEW fields only (checkpoint schema constraint: fields may
            # only be added) -- record the exact EyePACS/domain-aug config
            # this run used, so a checkpoint alone says whether it is a
            # v2a/v2b/v2c run without needing RUN_TAG or this notebook
            # alongside it.
            "eyepacs_5class_weight": float(EYEPACS_5CLASS_WEIGHT),
            "domain_aug": bool(DOMAIN_AUG),
            "eyepacs_max": int(EYEPACS_MAX) if USE_EYEPACS else None,
            "run_tag": RUN_TAG,
        }, CKPT_PATH)
        np.save(VAL_LOGITS_PATH, v_logits5)
        np.save(VAL_LABELS_PATH, v_labels)
        np.save(VAL_IDS_PATH, v_ids)
        np.save(VAL_BIN_LOGITS_PATH, v_logits_bin)
        print(f"           saved {os.path.basename(CKPT_PATH)} + val logits {tuple(v_logits5.shape)}")
    else:
        no_improve += 1

    # item C.3: last.pt written EVERY epoch regardless of improvement, so a
    # killed run can resume from here -- a separate file/schema from
    # CKPT_PATH, which only ever holds the best-so-far model for downstream use.
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "epoch": epoch,
        "best_qwk": float(best_qwk),
        "best_epoch": best_epoch,
        "history": history,
    }, LAST_CKPT_PATH)

    if no_improve >= PATIENCE:
        print(f"\nearly stop - no val QWK gain for {PATIENCE} epochs "
              f"(best epoch {best_epoch}, QWK {best_qwk:.4f})")
        break

print(f"\nbest epoch {best_epoch} | val QWK {best_qwk:.4f}")


## 10. Lock the binary-head threshold on VALIDATION (item 5)

Sweep the ROC curve on the validation split's binary-head logits from the
BEST checkpoint, take the first threshold whose sensitivity is `>=
TARGET_SENSITIVITY` (0.90), and report the specificity it actually achieves
at that point -- honestly, whatever it turns out to be. This is a validation-
only decision; the locked threshold is then APPLIED (not re-swept) on the
test split in the next cell, so the reported test specificity is a real
held-out number, not a re-optimized one.

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

_, _, val_logits_bin, val_labels, _ = evaluate(model, val_loader, criterion5, criterionBin)
val_probs_bin = 1.0 / (1.0 + np.exp(-val_logits_bin))
val_ref_true = (val_labels >= REFERABLE_FROM).astype(int)

fpr, tpr, thresholds = roc_curve(val_ref_true, val_probs_bin)
ok = tpr >= TARGET_SENSITIVITY
if not ok.any():
    if SMOKE:
        # verification-pass item 3: SMOKE must reach the last cell even on a
        # tiny val slice where 90% sensitivity may be unreachable. Fall back
        # to the max-sensitivity point instead of raising -- loudly, since
        # this threshold is NOT meaningful and must never be reused outside
        # SMOKE.
        idx = int(np.argmax(tpr))
        print(f"WARNING (SMOKE): no threshold on the validation ROC reaches "
              f"{TARGET_SENSITIVITY:.0%} sensitivity (max achieved: {tpr.max():.3f}) -- "
              f"continuing with the max-sensitivity threshold since SMOKE=True. This "
              f"threshold is MEANINGLESS; it exists only so the pipeline can be smoke-"
              f"tested end-to-end, never for a real run.")
    else:
        raise RuntimeError(
            f"No threshold on the validation ROC reaches {TARGET_SENSITIVITY:.0%} sensitivity "
            f"(max achieved: {tpr.max():.3f}). Do not lower TARGET_SENSITIVITY to make this pass -- "
            f"report the real max and treat it as a v2 training failure to fix (more epochs, "
            f"more referable oversampling, a different BINARY_LOSS_WEIGHT), not a threshold to relax.")
else:
    # roc_curve returns thresholds in DECREASING order, and tpr is
    # non-decreasing as the threshold falls -- so `ok` is True from the
    # FIRST index that clears the sensitivity bar through to the END of the
    # curve. The LAST True index (what this line used to take) is therefore
    # the LOWEST threshold on the whole curve: sensitivity ~1.0, specificity
    # ~0.0 -- the opposite of "most specific". The FIRST True index is the
    # HIGHEST threshold that still clears the sensitivity bar, i.e. the best
    # specificity available at that sensitivity floor -- that is the point
    # to lock.
    idx = int(np.argmax(ok))

LOCKED_THRESHOLD = float(thresholds[idx])
achieved_sens = float(tpr[idx])
achieved_spec = float(1 - fpr[idx])

print(f"locked binary threshold: {LOCKED_THRESHOLD:.4f}")
print(f"validation sensitivity at lock: {achieved_sens:.4f} (target >= {TARGET_SENSITIVITY:.0%})")
print(f"validation specificity at lock: {achieved_spec:.4f}  <- report this honestly, do not cherry-pick a different point")

if achieved_spec <= 0.05:
    print(f"WARNING: locked specificity is only {achieved_spec:.4f} at "
          f"{TARGET_SENSITIVITY:.0%} sensitivity -- this binary head is "
          f"functionally degenerate at this operating point (near-zero true-negative "
          f"rate, i.e. it is close to calling everything referable). Do not ship this "
          f"lock; treat it as a training failure to fix (more epochs, more referable "
          f"oversampling, a different BINARY_LOSS_WEIGHT), the same as the "
          f"no-threshold-reaches-target case above.")


## 11. Held-out test evaluation (item 7 + A.4 + C.5: all metrics -- including
a POOLED / IDRiD-only / APTOS-only breakdown and a live-path referable
number -- on v1's REAL recovered held-out split, threshold applied not
re-swept)


In [ ]:
def full_eval_report(labels, logits5, logits_bin, locked_threshold):
    """One evaluation, three referable-detection numbers side by side (item
    C.5): the plain 5-class argmax number, that SAME argmax plus the
    P(grade3)+P(grade4)>0.5 forces-referable safety check the live
    rule-engine also applies ("live-path"), and the separately-trained
    binary head's number at its locked threshold."""
    labels = np.asarray(labels)
    y_pred5 = logits5.argmax(axis=1)
    m = compute_metrics(labels, y_pred5)

    probs5 = np.exp(logits5 - logits5.max(axis=1, keepdims=True))
    probs5 = probs5 / probs5.sum(axis=1, keepdims=True)
    p34 = probs5[:, 3] + probs5[:, 4]
    live_path_pred = (y_pred5 >= REFERABLE_FROM) | (p34 > 0.5)
    ref_true = labels >= REFERABLE_FROM
    tp = int((ref_true & live_path_pred).sum())
    fn = int((ref_true & ~live_path_pred).sum())
    tn = int((~ref_true & ~live_path_pred).sum())
    fp = int((~ref_true & live_path_pred).sum())
    m["live_path_sens"] = _safe_rate(tp, tp + fn)
    m["live_path_spec"] = _safe_rate(tn, tn + fp)

    probs_bin = 1.0 / (1.0 + np.exp(-logits_bin))
    pred_bin = (probs_bin >= locked_threshold).astype(int)
    ref_true_i = ref_true.astype(int)
    tp = int(((ref_true_i == 1) & (pred_bin == 1)).sum())
    fn = int(((ref_true_i == 1) & (pred_bin == 0)).sum())
    tn = int(((ref_true_i == 0) & (pred_bin == 0)).sum())
    fp = int(((ref_true_i == 0) & (pred_bin == 1)).sum())
    m["binary_head_sens"] = _safe_rate(tp, tp + fn)
    m["binary_head_spec"] = _safe_rate(tn, tn + fp)

    m["confusion_matrix"] = confusion_matrix(labels, y_pred5, labels=list(range(NUM_CLASSES))).tolist()
    m["n"] = int(len(labels))
    return m


def print_eval_block(title, m):
    print(f"\n--- {title} (n={m['n']}) ---")
    print(f"QWK                                     : {m['qwk']:.4f}")
    print(f"5-class-head referable sens/spec        : {m['ref_sens']:.4f} / {m['ref_spec']:.4f}")
    print(f"+ P(g3)+P(g4)>0.5 safety-check sens/spec: {m['live_path_sens']:.4f} / {m['live_path_spec']:.4f}  "
          f"<- mirrors the live inference-time rule-engine path")
    print(f"binary-head sens/spec (locked {LOCKED_THRESHOLD:.4f})  : "
          f"{m['binary_head_sens']:.4f} / {m['binary_head_spec']:.4f}")
    print(f"grade-4 recall                          : {m['grade4_recall']:.4f}  (n={m['grade4_n']})")
    print(f"grade-1 recall                          : {m['grade1_recall']:.4f}  (n={m['grade1_n']})")
    print(f"mean signed error                       : {m['mean_signed_error']:+.4f}  (negative = under-grading)")
    print("confusion matrix (rows=truth, cols=pred):")
    print(pd.DataFrame(m["confusion_matrix"], index=[f"true{i}" for i in range(NUM_CLASSES)],
                       columns=[f"pred{i}" for i in range(NUM_CLASSES)]))


_, t_logits5, t_logits_bin, t_labels, t_ids = evaluate(model, test_loader, criterion5, criterionBin)

overall = full_eval_report(t_labels, t_logits5, t_logits_bin, LOCKED_THRESHOLD)
test_source = test_df["source"].values
assert len(test_source) == len(t_labels), (
    "test_df and the evaluated test set are out of order -- cannot slice by source. "
    "(test_loader uses shuffle=False, which should preserve row order; if this fires, "
    "something upstream reordered test_df after test_loader/test_ds were built.)")
by_source = {}
for src in ("idrid", "aptos"):
    mask = test_source == src
    by_source[src] = full_eval_report(t_labels[mask], t_logits5[mask], t_logits_bin[mask],
                                      LOCKED_THRESHOLD) if mask.any() else None

print("=" * 70)
print(f"v2 TEST RESULTS -- v1's REAL recovered held-out split (n={len(t_labels)})")
print("=" * 70)
print_eval_block("POOLED: APTOS + IDRiD recovered test", overall)
if by_source["idrid"] is not None:
    print_eval_block("IDRiD-ONLY recovered test", by_source["idrid"])
if by_source["aptos"] is not None:
    print_eval_block("APTOS-ONLY recovered test", by_source["aptos"])

# v1 comparison. item A.4: v1's own reported numbers are NOT all on the same
# population -- these are recomputed DIRECTLY from
# models/Model1/branchA_v1_test_logits.npy (+labels/ids), split by id prefix,
# not hand-copied from a report. Verification pass found and fixed one real
# mistake: V1_REFERENCE_IDRID_ONLY's grade4_recall used to hold 0.444, which
# is actually the POOLED grade-4 recall mislabeled as IDRiD-only -- the real
# IDRiD-only grade-4 recall is 0.300 (3/10). Each v1 number below is diffed
# against the v2 number computed on the SAME population.
V1_REFERENCE_POOLED = {"qwk": 0.8688, "ref_sens": 0.8603, "ref_spec": 0.9382}
V1_REFERENCE_IDRID_ONLY = {"qwk": 0.8242, "ref_sens": 0.8776, "ref_spec": 0.8966,
                           "grade4_recall": 0.3000, "grade1_recall": 0.0000}  # 3/10, 0/4 on IDRiD
V1_REFERENCE_APTOS_ONLY = {"qwk": 0.8727, "ref_sens": 0.8565, "ref_spec": 0.9419,
                           "grade4_recall": 0.4773, "grade1_recall": 0.6250}  # 21/44, 35/56 on APTOS

print("\nv1 -> v2 delta, POOLED (APTOS+IDRiD):")
for k, v1 in V1_REFERENCE_POOLED.items():
    v2 = overall[k]
    print(f"  {k:15s} v1={v1:.4f}  v2={v2:.4f}  delta={v2 - v1:+.4f}")

if by_source["idrid"] is not None:
    print("\nv1 -> v2 delta, IDRiD-ONLY:")
    for k, v1 in V1_REFERENCE_IDRID_ONLY.items():
        v2 = by_source["idrid"][k]
        print(f"  {k:15s} v1={v1:.4f}  v2={v2:.4f}  delta={v2 - v1:+.4f}")

if by_source["aptos"] is not None:
    print("\nv1 -> v2 delta, APTOS-ONLY:")
    for k, v1 in V1_REFERENCE_APTOS_ONLY.items():
        v2 = by_source["aptos"][k]
        print(f"  {k:15s} v1={v1:.4f}  v2={v2:.4f}  delta={v2 - v1:+.4f}")

metrics_out = {
    "run_tag": RUN_TAG, "use_eyepacs": USE_EYEPACS, "smoke": SMOKE,
    "eyepacs_5class_weight": EYEPACS_5CLASS_WEIGHT, "domain_aug": DOMAIN_AUG,
    "eyepacs_max": EYEPACS_MAX if USE_EYEPACS else None,
    "test_pooled": overall, "test_idrid_only": by_source["idrid"], "test_aptos_only": by_source["aptos"],
    "val_at_lock": {"sensitivity": achieved_sens, "specificity": achieved_spec},
    "v1_reference_pooled": V1_REFERENCE_POOLED, "v1_reference_idrid_only": V1_REFERENCE_IDRID_ONLY,
    "v1_reference_aptos_only": V1_REFERENCE_APTOS_ONLY,
    "img_size": IMG_SIZE, "grade4_weight_boost": GRADE4_WEIGHT_BOOST,
    "best_epoch": best_epoch, "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
}
with open(METRICS_PATH, "w") as f:
    json.dump(metrics_out, f, indent=2, default=float)

np.save(TEST_LOGITS_PATH, t_logits5)
np.save(TEST_LABELS_PATH, t_labels)
np.save(TEST_IDS_PATH, t_ids)
np.save(TEST_BIN_LOGITS_PATH, t_logits_bin)
print(f"\nsaved {METRICS_PATH}")


## 12. Artifacts to download -- AND THE MANDATORY NEXT STEP (item 8)

Download every file below from the Kaggle "Output" panel. Then, **before
this model touches anything downstream of a raw checkpoint**:

1. Update `preprocessModel1.m`, `diagnostics/MODEL_INTERFACE_REFERENCE.md`,
   and `docs/model-handoff-guide.md` to say **512** (or whatever `IMG_SIZE`
   actually was for the run you are shipping), not 384. Do this from the
   checkpoint's own `img_size` field, not from memory -- that is exactly how
   the 384-vs-512(claimed as 512 in a stale doc, actually 384) confusion
   happened the first time.
2. Re-run `exportModel1Predictions.py` equivalent against v2's saved val/test
   logits (already in the right shape/format above) and then
   **`calibrateBranchA.m` with `opts.imgSize` set to v2's real resolution**
   -- required, no default, will error otherwise (see that file's header).
   This OVERWRITES `calibration_v1.json`'s `qhat=0.8432`, which was fitted
   for v1 at 384px and means nothing for this model.
3. Confirm the overwrite worked: `branchAInfer.py`'s `load_calibration(ckpt)`
   and `branchAInferMatlab.m` both carry a version guard added specifically
   for this (2026-09-19) that REFUSES to apply a calibration file whose
   `trainedImgSize` doesn't match the loaded model -- if you skip step 2, the
   guard will correctly report the model as UNCALIBRATED rather than
   silently using v1's qhat. Do not treat that warning as a bug to route
   around; it is the safety net this exact mistake needs.

In [ ]:
print("Download from the Kaggle 'Output' panel after the run:\n")
for p in [CKPT_PATH, LAST_CKPT_PATH, HISTORY_PATH, VAL_LOGITS_PATH, VAL_LABELS_PATH, VAL_IDS_PATH,
          VAL_BIN_LOGITS_PATH, TEST_LOGITS_PATH, TEST_LABELS_PATH, TEST_IDS_PATH,
          TEST_BIN_LOGITS_PATH, METRICS_PATH]:
    print(" ", p)
print("\nThen: update preprocessModel1.m / MODEL_INTERFACE_REFERENCE.md / model-handoff-guide.md")
print("      to IMG_SIZE, and re-run calibrateBranchA.m with opts.imgSize=IMG_SIZE.")
print("      calibration_v1.json's qhat=0.8432 is NOT valid for this model until then.")

if SMOKE:
    print("\n" + "=" * 70)
    print("SMOKE RUN COMPLETE - numbers are meaningless")
    print("=" * 70)
